# Extensive Exploratory Data Analysis (EDA) - PGS S6E9
This notebook focuses purely on deep-dive Exploratory Data Analysis for the **Kaggle Playground Series S6E9: Predicting EV Purchase**.
The goal is to understand data quality, distributions, anomalies, and feature interactions in detail before moving to feature engineering and modeling.

### Table of Contents:
1. Setup & Data Loading
2. Data Quality Check (Missing Values & Duplicates)
3. Summary Statistics
4. Univariate Analysis
5. Bivariate Analysis (Features vs Target)
6. Multivariate Analysis & Correlations
7. Hypotheses & Takeaways


## 1. Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

import warnings
warnings.filterwarnings('ignore')

# Set plot aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

In [ ]:
# Load data
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e9'
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train dataset shape: {train.shape}")
print(f"Test dataset shape: {test.shape}")
display(train.head())

## 2. Data Quality Check

In [ ]:
# General info
train.info()

In [ ]:
# Missing values check
missing_train = train.isnull().sum()
missing_test = test.isnull().sum()

missing_df = pd.DataFrame({
    'Train Missing': missing_train[missing_train > 0],
    'Test Missing': missing_test[missing_test > 0]
})
print("Missing Values Summary:")
display(missing_df)

if missing_df.empty:
    print("No missing values found in both datasets!")

In [ ]:
# Duplicate check
train_dupes = train.duplicated().sum()
test_dupes = test.duplicated().sum()
print(f"Duplicates in Train: {train_dupes}")
print(f"Duplicates in Test: {test_dupes}")

In [ ]:
# Unique values per column to identify categorical vs continuous
unique_counts = train.nunique()
print("Unique values per column:\n", unique_counts)

## 3. Summary Statistics

In [ ]:
# Numerical features
num_cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 
            'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
display(train[num_cols].describe().T)

In [ ]:
# Categorical features
cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 
            'Subsidy_Available', 'Range_Anxiety_Level']
display(train[cat_cols].describe().T)

## 4. Univariate Analysis

### Target Distribution

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=train, x='Will_Buy_EV', palette='Set2')
plt.title('Distribution of Target Variable (Will_Buy_EV)')

# Add percentage labels
total = len(train['Will_Buy_EV'])
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')
    
plt.show()

### Numerical Features Distribution

In [ ]:
fig, axes = plt.subplots(math.ceil(len(num_cols)/2), 2, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(train[col], kde=True, ax=axes[i], bins=40, color='skyblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_ylabel('Frequency')

# Remove any empty subplots
for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()

### Boxplots for Outlier Detection

In [ ]:
fig, axes = plt.subplots(math.ceil(len(num_cols)/3), 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(y=train[col], ax=axes[i], color='lightgreen')
    axes[i].set_title(f'Boxplot of {col}')

for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()

### Categorical Features Distribution

In [ ]:
fig, axes = plt.subplots(math.ceil(len(cat_cols)/2), 2, figsize=(16, 15))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=train, y=col, ax=axes[i], palette='pastel', order=train[col].value_counts().index)
    axes[i].set_title(f'Count of {col}')
    axes[i].set_xlabel('Count')
    axes[i].set_ylabel(col)
    
for j in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()

## 5. Bivariate Analysis (Features vs Target)

### Numerical Features vs Target

In [ ]:
fig, axes = plt.subplots(math.ceil(len(num_cols)/2), 2, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.violinplot(data=train, x='Will_Buy_EV', y=col, ax=axes[i], palette='Set2', inner='quartile')
    axes[i].set_title(f'{col} distribution by Target')

for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()

### Categorical Features vs Target

In [ ]:
fig, axes = plt.subplots(math.ceil(len(cat_cols)/2), 2, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # Calculate proportions to see relative impact
    prop_df = train.groupby(col)['Will_Buy_EV'].value_counts(normalize=True).rename('Proportion').reset_index()
    sns.barplot(data=prop_df, x=col, y='Proportion', hue='Will_Buy_EV', ax=axes[i], palette='Set2')
    axes[i].set_title(f'Proportion of Target by {col}')
    axes[i].tick_params(axis='x', rotation=45)

for j in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()

## 6. Multivariate Analysis & Correlations

In [ ]:
# Map target to 1/0 for correlation analysis
train['target_num'] = train['Will_Buy_EV'].map({'Yes': 1, 'No': 0})

# Calculate correlation matrix
corr_cols = num_cols + ['target_num']
corr_matrix = train[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix of Numerical Features', fontsize=16)
plt.show()

### Pairplot for Top Correlated Features

In [ ]:
# Select a few interesting features based on correlations (or intuition)
top_features = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Charging_Stations_Near_Home']

# Sample data if dataset is too large to pairplot quickly
sample_df = train.sample(n=min(10000, len(train)), random_state=42)

sns.pairplot(sample_df[top_features + ['Will_Buy_EV']], hue='Will_Buy_EV', palette='Set2', diag_kind='kde', corner=True)
plt.suptitle('Pairplot of Key Numerical Features', y=1.02, fontsize=16)
plt.show()